# UK accent baseline: Whisper on OpenSLR 83

Transcribes UK and Irish English speech with four sizes of Whisper and measures word error rate by accent, with speaker-level confidence intervals.

**On Colab:** Runtime → Change runtime type → **T4 GPU**, then run the cells in order. Running locally works too (CUDA or Apple Silicon); start Jupyter from the repo.

## 1. Setup

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/uk-accent-asr"):
        !git clone -q https://github.com/temialao/uk-accent-asr.git /content/uk-accent-asr
    os.chdir("/content/uk-accent-asr")
    !pip install -q jiwer whisper-normalizer   # everything else is preinstalled on Colab
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (this will be slow)")

In [ ]:
# Optional: keep results in Google Drive so a dropped Colab session can resume where it stopped.
SAVE_TO_DRIVE = False

BASE = "."
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/uk-accent-asr"
QUICK, FULL = f"{BASE}/results_quick", f"{BASE}/results"
print(QUICK, FULL)

In [ ]:
!python -m pytest -q

## 2. What's in the dataset

Each config is one accent and gender. Some groups have very few speakers; the report flags them.

In [ ]:
from ukasr import data
configs = data.list_configs()
configs

## 3. Quick run: one model, 100 clips per accent/gender

Checks everything works end to end. The first run downloads the dataset (several GB), so allow a few minutes.

In [ ]:
!python -m ukasr.run_baseline --models openai/whisper-small.en --max-per-config 100 --out {QUICK}

In [ ]:
import subprocess
from IPython.display import Image, Markdown, display

def show(results):
    subprocess.run([sys.executable, "-m", "ukasr.report", "--results", results], check=True)
    display(Image(f"{results}/wer_by_accent.png"))
    summary = open(f"{results}/summary.md").read().replace("![WER by accent](wer_by_accent.png)", "")
    display(Markdown(summary))

show(QUICK)

## 4. Full baseline: four model sizes, up to 400 clips per accent/gender

Clips are spread evenly across speakers, and every model sees the same ones. Expect roughly 20–40 minutes on a T4, depending on the GPU you're given. If the session drops, re-run this cell: finished model/accent pairs are skipped.

In [ ]:
# Default models: whisper-tiny.en, base.en, small.en and large-v3-turbo
!python -m ukasr.run_baseline --max-per-config 400 --out {FULL}

In [ ]:
show(FULL)

## 5. Listen to the worst Northern clips

This is where local knowledge helps: note *why* each one failed (vowel, place name, dialect word, recording quality). Those notes become the error analysis in the README.

In [ ]:
import pandas as pd
from IPython.display import Audio
from datasets import Audio as HFAudio, load_dataset

MODEL, ACCENT, N = "whisper-small.en", "northern", 10
worst = pd.read_csv(f"{FULL}/worst/{MODEL}__{ACCENT}.csv").head(N)

for config, group in worst.groupby("config"):
    ds = load_dataset(data.DATASET_ID, config, split="train").cast_column("audio", HFAudio(decode=False))
    row_of = {line_id: i for i, line_id in enumerate(ds["line_id"])}
    for _, clip in group.iterrows():
        print(f"{config}  speaker {clip.speaker_id}  WER {clip.wer:.0%}")
        print(f"  said:  {clip.ref}\n  heard: {clip.hyp}")
        display(Audio(data.decode_audio(ds[row_of[clip.line_id]]["audio"]), rate=data.TARGET_SR))

## 6. Save the results

Download the zip, unzip it into the repo's `results/` folder on your machine, then run
`python -m ukasr.report --update-readme` and commit. (Results include transcripts from OpenSLR 83, CC BY-SA 4.0.)

In [ ]:
import shutil
shutil.make_archive("results", "zip", FULL)
if IN_COLAB:
    from google.colab import files
    files.download("results.zip")